# 05 — Task 5: Creative challenge — improving FER2013 (25 %)

Starting from the best Task 1 configuration we test, in order, groups of improvements. Inside
each group the options are compared on the **validation** set (PublicTest) and the best one is
kept only if it improves validation macro-F1 (greedy forward selection). PrivateTest numbers
are logged for every row but **never** used for a decision.

| group | strategy (assignment list) | options |
|---|---|---|
| A0 | feature engineering | gray vs CLAHE input (from the Task 1 preprocessing study) |
| A1 | feature engineering | HOG cell size / number of orientations |
| A2 | class balancing | none vs class weights vs SMOTE (`imblearn` pipeline, no leakage) |
| A3 | data augmentation | none vs horizontal flip vs flip + small rotations (train only) |
| B  | **KDEF-derived features** | geometric features: none / all 20 / the subset that is discriminative on KDEF *and* transfers to FER |
| C1 | **KDEF emotion prototypes** | Procrustes distance of each face to the 7 KDEF prototype shapes |
| C2 | data curation & label verification | remove duplicates/blank images; also drop training images whose label conflicts with both a cross-validated model and the KDEF prototypes |
| D  | **merge FER + KDEF** (paper: Kırbız, *Improving FER through dataset merging and balanced training*) | add the KDEF training subjects (clean / FER-like degraded) with inverse-frequency class weights |
| A4 | feature selection | none vs SelectKBest (ANOVA) vs PCA (95 % variance) |
| A5 | hyper-parameter tuning | finer grid around the current parameters |

To keep the ~25 trainings affordable, the ablations use a stratified subset of `ABL_N` training
images; the final configuration is re-trained on the full training set and evaluated once.

In [ ]:
# Common setup + evaluation helpers (identical in every notebook of the project)
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import StratifiedShuffleSplit

sys.path.insert(0, str(Path.cwd()))
try:
    from src import features as F
except ImportError:  # fuzzy environment (no OpenCV/dlib): only cached features are used there
    from types import SimpleNamespace
    F = SimpleNamespace(EMOTIONS=["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"],
                        CACHE_DIR=Path("data") / "cache")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# SMOKE=1 (environment variable) runs everything on small subsets with reduced grids, to
# check the notebook end-to-end in minutes. The delivered results use SMOKE=0 (default).
SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
N_JOBS = -1
EMO = F.EMOTIONS
RESULTS_DIR, FIG_DIR, MODEL_DIR = Path("results"), Path("figures"), Path("models")
for d in (RESULTS_DIR, FIG_DIR, MODEL_DIR, F.CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "savefig.bbox": "tight"})
print(f"SMOKE={SMOKE}")


def metrics(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred),
            "bal_acc": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", labels=range(7), zero_division=0)}


def per_class_f1(y_true, y_pred):
    return pd.Series(f1_score(y_true, y_pred, average=None, labels=range(7), zero_division=0), index=EMO)


_RESULTS = {}


def reset_results(task):
    """Start a fresh results/<task>.csv for this run of the notebook."""
    _RESULTS[task] = []
    (RESULTS_DIR / f"{task}.csv").unlink(missing_ok=True)


def log_result(task, y_true, y_pred, **info):
    """Append one evaluation row (fixed schema) to results/<task>.csv and return it."""
    row = {**info, **{k: round(v, 4) for k, v in metrics(y_true, y_pred).items()}, "n": len(y_true)}
    _RESULTS.setdefault(task, []).append(row)
    pd.DataFrame(_RESULTS[task]).to_csv(RESULTS_DIR / f"{task}.csv", index=False)
    return row


def fixed_table(df, index, columns, value, index_order, column_order):
    """Pivot to a table with a FIXED row/column layout (same x and y in every task)."""
    t = df.pivot_table(index=index, columns=columns, values=value, aggfunc="first")
    return t.reindex(index=index_order, columns=column_order)


def save_table(df, name, float_format="%.3f"):
    """Save a table as CSV and as a LaTeX tabular (pasted into report/*.md)."""
    df.to_csv(RESULTS_DIR / f"{name}.csv")
    with open(RESULTS_DIR / f"{name}.tex", "w", encoding="utf-8") as f:
        f.write(df.to_latex(float_format=float_format, na_rep="--"))
    return df


def plot_cm(y_true, y_pred, title="", ax=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=range(7), normalize="true" if normalize else None)
    ax = ax or plt.subplots(figsize=(4.2, 3.6))[1]
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues", cbar=False,
                xticklabels=[e[:3] for e in EMO], yticklabels=[e[:3] for e in EMO], ax=ax,
                vmin=0, vmax=1 if normalize else None, annot_kws={"size": 7})
    ax.set_title(title, fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    return ax


def plot_cm_grid(items, fname, ncols=4):
    """items: list of (title, y_true, y_pred). Saved in figures/ for the report appendix."""
    nrows = int(np.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.7 * nrows), squeeze=False)
    for ax, (title, yt, yp) in zip(axes.ravel(), items):
        plot_cm(yt, yp, title, ax=ax)
    for ax in axes.ravel()[len(items):]:
        ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / fname); plt.show()


def stratified_subset(y, n, seed=SEED):
    """Indices of a stratified random subset of size n (all indices if n >= len(y))."""
    if n is None or n >= len(y):
        return np.arange(len(y))
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    return np.sort(next(sss.split(np.zeros(len(y)), y))[0])


def scores(model, X):
    """Class scores for confidence analyses (predict_proba, else softmax(decision_function))."""
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X)
        except AttributeError:
            pass
    d = model.decision_function(X)
    e = np.exp(d - d.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def save_bundle(name, model, spec, **extra):
    """Persist a FITTED model together with the feature spec needed to use it (demo, other tasks)."""
    bundle = {"model": model, "spec": spec, "labels": EMO, **extra}
    joblib.dump(bundle, MODEL_DIR / f"{name}.pkl", compress=3)
    print(f"saved models/{name}.pkl")


def show_images(imgs, titles=None, ncols=8, size=1.5, fname=None, suptitle=None, cmap="gray"):
    nrows = int(np.ceil(len(imgs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * 1.15 * nrows), squeeze=False)
    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < len(imgs):
            ax.imshow(imgs[i], cmap=None if imgs[i].ndim == 3 else cmap, vmin=0, vmax=255)
            if titles is not None:
                ax.set_title(titles[i], fontsize=7)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    if fname:
        fig.savefig(FIG_DIR / fname)
    plt.show()

In [ ]:
# Classifiers, feature sets and the tuning protocol (shared by notebooks 01, 02, 03, 05)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit, GroupKFold
from sklearn.base import clone

MODELS = ["LR", "SVM", "RF"]               # fixed row order of every result grid
FEATURES = ["HOG", "LM", "PCA", "HOG+LM"]   # fixed column order of every result grid
FEATURE_BLOCKS = {"HOG": ["hog"], "LM": ["lm"], "PCA": ["pix"], "HOG+LM": ["hog", "lm"]}
N_TREES = 100 if SMOKE else 300


def base_estimator(model, n_jobs=1):
    if model == "LR":
        return LogisticRegression(max_iter=3000, random_state=SEED)
    if model == "SVM":
        return SVC(kernel="rbf", gamma="scale", cache_size=2000, random_state=SEED)
    if model == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", n_jobs=n_jobs, random_state=SEED)
    raise ValueError(model)


def make_pipeline(model, feature, n_jobs=1):
    """StandardScaler -> [PCA (eigenfaces, only for raw pixels)] -> classifier."""
    steps = [("scaler", StandardScaler())]
    if feature == "PCA":
        steps.append(("pca", PCA(n_components=100, whiten=True, random_state=SEED)))
    steps.append(("clf", base_estimator(model, n_jobs)))
    return Pipeline(steps)


if SMOKE:
    GRIDS = {"LR": {"clf__C": [0.1, 1.0], "clf__class_weight": ["balanced"]},
             "SVM": {"clf__C": [1, 10], "clf__class_weight": ["balanced"]},
             "RF": {"clf__max_depth": [None], "clf__class_weight": [None, "balanced_subsample"]}}
    PCA_GRID = {"pca__n_components": [60]}
else:
    GRIDS = {"LR": {"clf__C": [0.01, 0.1, 1.0], "clf__class_weight": [None, "balanced"]},
             "SVM": {"clf__C": [1, 3, 10], "clf__class_weight": [None, "balanced"]},
             "RF": {"clf__max_depth": [None, 25], "clf__min_samples_leaf": [1, 3],
                    "clf__class_weight": [None, "balanced_subsample"]}}
    PCA_GRID = {"pca__n_components": [60, 120]}


def param_grid(model, feature):
    return {**GRIDS[model], **(PCA_GRID if feature == "PCA" else {})}


def tune_predefined(model, feature, X_tr, y_tr, X_va, y_va, n_search):
    """Grid search: fit on (a stratified subset of) Training, score macro-F1 on PublicTest."""
    idx = stratified_subset(y_tr, n_search)
    X = np.vstack([X_tr[idx], X_va]); y = np.concatenate([y_tr[idx], y_va])
    ps = PredefinedSplit(np.concatenate([-np.ones(len(idx)), np.zeros(len(y_va))]))
    gs = GridSearchCV(make_pipeline(model, feature), param_grid(model, feature),
                      scoring="f1_macro", cv=ps, n_jobs=N_JOBS, refit=False)
    gs.fit(X, y)
    return gs.best_params_, gs.best_score_


def tune_groupkfold(model, feature, X, y, groups, n_splits=5):
    """Subject-independent CV (no validation split exists for KDEF)."""
    gs = GridSearchCV(make_pipeline(model, feature), param_grid(model, feature), scoring="f1_macro",
                      cv=GroupKFold(n_splits=n_splits), n_jobs=N_JOBS, refit=False)
    gs.fit(X, y, groups=groups)
    std = gs.cv_results_["std_test_score"][gs.best_index_]
    return gs.best_params_, gs.best_score_, std


def fit_final(model, feature, params, X, y, sample_weight=None):
    pipe = make_pipeline(model, feature, n_jobs=N_JOBS).set_params(**params)
    fit_kw = {"clf__sample_weight": sample_weight} if sample_weight is not None else {}
    return pipe.fit(X, y, **fit_kw)

In [ ]:
# Cached datasets and features (written by 00_data_preparation.ipynb)
# Dataset keys: "fer_gray", "fer_aligned", "kdef_m0.0_gray" ... "kdef_m0.3_gray", "kdef_m0.2_rgb",
# "kdef_m0.2_gray_aligned". Splits: FER train/val/test, KDEF train/test.
import re
FEATURE_BLOCKS.update({"GEO": ["geo"]})
_DS = {}


def images(key, split):
    if key not in _DS:
        _DS[key] = dict(np.load(F.CACHE_DIR / f"{key}.npz", allow_pickle=True))
    return _DS[key][f"X_{split}"], _DS[key][f"y_{split}"]


def meta(key, field, split):
    images(key, split)
    return _DS[key][f"{field}_{split}"]


def crop_margin(key):
    """Crop margin of a dataset (KDEF keys encode it, e.g. 'kdef_m0.2_gray'; FER crops: 0)."""
    m = re.search(r"_m([0-9.]+)_", key)
    return float(m.group(1)) if m else 0.0


def landmarks(key, split, variant="gray"):
    X, _ = images(key, split)
    vt = "" if variant == "gray" else f"-{variant}"
    return F.cached(f"{key}{vt}_{split}_pts",
                    lambda: F.extract_landmarks(F.apply_variant(X, variant), margin=crop_margin(key)))


def features(key, split, feature, variant="gray", hog=None):
    """Feature matrix of a cached dataset split (computed once, then loaded from data/cache)."""
    if feature == "HOG+LM":
        return np.hstack([features(key, split, "HOG", variant, hog), features(key, split, "LM", variant)])
    X, _ = images(key, split)
    vt = "" if variant == "gray" else f"-{variant}"
    ht = "" if not hog else "_o{}c{}".format(hog["orientations"], hog["pixels_per_cell"][0])
    spec = {"variant": variant, "blocks": FEATURE_BLOCKS[feature], "hog": hog or {}}
    pts = landmarks(key, split, variant) if feature in ("LM", "GEO") else None
    return F.cached(f"{key}{vt}_{split}_{feature}{ht}", lambda: F.extract_feature_set(X, spec, pts=pts))


def spec_for(feature, variant="gray", hog=None, crop_margin=0.0):
    """Feature spec stored in a model bundle (lets demo_emotion.py rebuild the features)."""
    blocks = sum((FEATURE_BLOCKS[f] for f in feature.split("+")), [])
    return {"variant": variant, "blocks": blocks, "hog": hog or {}, "feature": feature, "crop_margin": crop_margin}

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from scipy.stats import spearmanr

reset_results("task5_ablation"); reset_results("task5_final")
t1 = json.load(open(RESULTS_DIR / "task1_best.json"))
MODEL, FEAT, PARAMS = t1["model"], t1["feature"], t1["params"]
print("Start: Task 1 best =", MODEL, FEAT, PARAMS)

y_tr_full, y_va_full, y_te = images("fer_gray", "train")[1], images("fer_gray", "val")[1], images("fer_gray", "test")[1]
VA = stratified_subset(y_va_full, 1000 if SMOKE else None)
y_va = y_va_full[VA]
ABL_N = 2000 if SMOKE else 12000
ABL_IDX = stratified_subset(y_tr_full, ABL_N)
FULL_IDX = stratified_subset(y_tr_full, 4000 if SMOKE else None)
FINAL_MAX_N = 6000 if SMOKE else 60000      # cap on the final training size (SVM training is O(n^2))
HOG_DEFAULT = {"orientations": 9, "pixels_per_cell": (8, 8), "cells_per_block": (2, 2)}

## Building blocks: feature construction, training set, pipeline

In [ ]:
KDEF_KEY = "kdef_m0.2_gray"
kd_pts_tr = landmarks(KDEF_KEY, "train")
y_kd_tr, y_kd_te = images(KDEF_KEY, "train")[1], images(KDEF_KEY, "test")[1]
MEAN_SHAPE, PROTOTYPES = F.build_prototypes(kd_pts_tr, y_kd_tr)       # KDEF emotion prototypes


def build_X(cfg, key, split):
    """Feature matrix for a dataset split under a config (blocks in a fixed order)."""
    v, parts = cfg["variant"], []
    for b in cfg["blocks"]:
        if b == "hog":
            parts.append(features(key, split, "HOG", v, cfg["hog"] if cfg["hog"] != HOG_DEFAULT else None))
        elif b == "lm":
            parts.append(features(key, split, "LM", v))
        elif b == "pix":
            parts.append(features(key, split, "PCA", v))
        elif b == "geo":
            parts.append(features(key, split, "GEO", v)[:, cfg["geo_idx"]])
        elif b == "proto":
            parts.append(F.prototype_distances(landmarks(key, split, v), MEAN_SHAPE, PROTOTYPES))
    return np.hstack(parts)


def make_aug_dataset(mode):
    """FER with an augmented training split (val/test untouched). Keeps the origin of every row."""
    name = f"fer_gray_aug_{mode}"
    if not (F.CACHE_DIR / f"{name}.npz").exists():
        X, y = images("fer_gray", "train")
        Xa, ya, origin = F.augment_images(X, y, mode=mode, seed=SEED)
        d = {"X_train": Xa, "y_train": ya, "origin_train": origin}
        for s in ("val", "test"):
            d[f"X_{s}"], d[f"y_{s}"] = images("fer_gray", s)
        np.savez_compressed(F.CACHE_DIR / f"{name}.npz", **d)
    return name


def make_degraded_kdef():
    name = f"{KDEF_KEY}_deg"
    if not (F.CACHE_DIR / f"{name}.npz").exists():
        d = dict(np.load(F.CACHE_DIR / f"{KDEF_KEY}.npz", allow_pickle=True))
        d["X_train"], d["X_test"] = F.degrade_batch(d["X_train"], SEED), F.degrade_batch(d["X_test"], SEED + 1)
        np.savez_compressed(F.CACHE_DIR / f"{name}.npz", **d)
    return name


def training_set(cfg, base_idx, max_n=None):
    """Rows used for training under cfg: subset → curation → augmentation → (+KDEF)."""
    idx = np.asarray(base_idx)
    if cfg["curation"] in ("dedup", "dedup+flags"):
        idx = idx[~np.isin(idx, DROP_DEDUP)]
    if cfg["curation"] == "dedup+flags":
        idx = idx[~np.isin(idx, FLAGGED)]
    if cfg["aug"] == "none":
        X, y = build_X(cfg, "fer_gray", "train")[idx], y_tr_full[idx]
    else:
        key = make_aug_dataset(cfg["aug"])
        rows = np.where(np.isin(meta(key, "origin", "train"), idx))[0]
        X, y = build_X(cfg, key, "train")[rows], images(key, "train")[1][rows]
    if max_n is not None and len(y) > max_n:
        keep = stratified_subset(y, max_n); X, y = X[keep], y[keep]
    if cfg["merge"] != "none":
        kkey = make_degraded_kdef() if cfg["merge"] == "kdef_degraded" else KDEF_KEY
        X = np.vstack([X, build_X(cfg, kkey, "train")]); y = np.concatenate([y, y_kd_tr])
    return X, y


def make_model(cfg, n_features):
    steps = [("scaler", StandardScaler())]
    if "pix" in cfg["blocks"]:
        steps.append(("pca", PCA(n_components=cfg["params"].get("pca__n_components", 100), whiten=True, random_state=SEED)))
    if cfg["select"] == "kbest":
        steps.append(("select", SelectKBest(f_classif, k=max(10, n_features // 2))))
    elif cfg["select"] == "pca95":
        steps.append(("select", PCA(n_components=0.95, random_state=SEED)))
    if cfg["balance"] == "smote":
        steps.append(("smote", SMOTE(random_state=SEED)))
    clf = base_estimator(MODEL, N_JOBS).set_params(**{k[5:]: v for k, v in cfg["params"].items() if k.startswith("clf__")})
    if "class_weight" in clf.get_params():
        cw = ("balanced_subsample" if MODEL == "RF" else "balanced") if cfg["balance"] == "cw" else None
        clf.set_params(class_weight=cw)
    steps.append(("clf", clf))
    return ImbPipeline(steps)


def evaluate(cfg, group, option, base_idx=ABL_IDX, table="task5_ablation", max_n=None):
    t = time.time()
    X, y = training_set(cfg, base_idx, max_n)
    model = make_model(cfg, X.shape[1]).fit(X, y)
    X_va, X_te = build_X(cfg, "fer_gray", "val")[VA], build_X(cfg, "fer_gray", "test")
    r_va = log_result(table, y_va, model.predict(X_va), group=group, option=option, split="val", n_train=len(y), dims=X.shape[1])
    p_te = model.predict(X_te)
    log_result(table, y_te, p_te, group=group, option=option, split="test", n_train=len(y), dims=X.shape[1])
    if cfg["merge"] != "none" or group == "D":
        log_result(table, y_kd_te, model.predict(build_X(cfg, KDEF_KEY, "test")), group=group, option=option,
                   split="kdef_test", n_train=len(y), dims=X.shape[1])
    print(f"  {group:3s} {option:28s} val F1={r_va['macro_f1']:.3f}  n={len(y)} d={X.shape[1]}  ({time.time() - t:.0f}s)")
    return r_va["macro_f1"], model, p_te


CFG = {"variant": "gray", "blocks": FEATURE_BLOCKS[FEAT], "hog": HOG_DEFAULT, "geo_idx": None,
       "curation": "none", "aug": "none", "merge": "none", "select": "none",
       "balance": "cw" if PARAMS.get("clf__class_weight") else "none", "params": dict(PARAMS)}
HISTORY = []


def try_group(group, options):
    """Evaluate each option (a dict of cfg changes); adopt the best if it beats the current val F1."""
    global CFG, BEST_VAL
    res = {}
    for name, change in options.items():
        cfg = {**CFG, **change}
        res[name] = (evaluate(cfg, group, name)[0], cfg)
    best = max(res, key=lambda n: res[n][0])
    if res[best][0] > BEST_VAL + 0.002 and res[best][1] != CFG:
        CFG, BEST_VAL = res[best][1], res[best][0]
        HISTORY.append((group, best, BEST_VAL)); print(f"  → adopted '{best}' (val F1 {BEST_VAL:.3f})")
    else:
        print(f"  → kept current configuration (val F1 {BEST_VAL:.3f})")


DROP_DEDUP, FLAGGED = np.array([], int), np.array([], int)
BEST_VAL = evaluate(CFG, "base", f"Task 1 best ({MODEL}/{FEAT})")[0]

## A0–A3: input variant, HOG parameters, class balancing, augmentation

In [ ]:
try_group("A0", {"gray": {"variant": "gray"}, "CLAHE": {"variant": "clahe"}})
if "hog" in CFG["blocks"]:
    try_group("A1", {f"HOG o{o} c{c}": {"hog": {"orientations": o, "pixels_per_cell": (c, c), "cells_per_block": (2, 2)}}
                     for o in (9, 12) for c in (8, 6)})
try_group("A2", {"no balancing": {"balance": "none"}, "class weights": {"balance": "cw"}, "SMOTE": {"balance": "smote"}})
try_group("A3", {"no augmentation": {"aug": "none"}, "flip": {"aug": "flip"}, "flip + rotation": {"aug": "flip_rot"}})

## B: geometric features selected on KDEF
Each of the 20 geometric features is scored on KDEF by (1) its mutual information with the
emotion and (2) its **transfer consistency**: the Spearman correlation between its per-emotion
means on KDEF and on FER. A feature is kept if it is informative on clean data *and* behaves
the same way in the wild.

In [ ]:
G_kd, G_fer = features(KDEF_KEY, "train", "GEO"), features("fer_gray", "train", "GEO")
mi = mutual_info_classif(G_kd, y_kd_tr, random_state=SEED)
mean_kd = pd.DataFrame(G_kd).groupby(y_kd_tr).mean().values
mean_fer = pd.DataFrame(G_fer).groupby(y_tr_full).mean().values
consistency = np.array([spearmanr(mean_kd[:, j], mean_fer[:, j])[0] for j in range(G_kd.shape[1])])
geo_rank = pd.DataFrame({"KDEF mutual information": mi, "KDEF→FER consistency (Spearman)": consistency},
                        index=F.GEO_NAMES).sort_values("KDEF mutual information", ascending=False)
GEO_SEL = [F.GEO_NAMES.index(n) for n in geo_rank.index if geo_rank.loc[n, "KDEF→FER consistency (Spearman)"] >= 0.6][:10]
geo_rank["selected"] = [F.GEO_NAMES.index(n) in GEO_SEL for n in geo_rank.index]
save_table(geo_rank, "task5_geo_ranking"); display(geo_rank.round(3))

try_group("B", {"no geometric": {},
                "+ all 20 geometric": {"blocks": [b for b in CFG["blocks"] if b != "geo"] + ["geo"], "geo_idx": list(range(20))},
                f"+ {len(GEO_SEL)} KDEF-selected geometric": {"blocks": [b for b in CFG["blocks"] if b != "geo"] + ["geo"], "geo_idx": GEO_SEL}})

## C1: KDEF emotion prototypes
The KDEF training faces are Procrustes-aligned (translation, scale, rotation removed) and averaged
per emotion. The distance of a FER face to each of the 7 prototype shapes is a compact,
identity-normalised description of "which posed expression this face looks like".

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(14, 2.3))
for ax, c in zip(axes, range(7)):
    ax.scatter(MEAN_SHAPE[:, 0], -MEAN_SHAPE[:, 1], s=4, c="lightgray")
    ax.scatter(PROTOTYPES[c][:, 0], -PROTOTYPES[c][:, 1], s=4, c="C3")
    ax.set_title(EMO[c], fontsize=9); ax.axis("off"); ax.set_aspect("equal")
fig.suptitle("KDEF emotion prototypes (red) vs mean shape (gray)", fontsize=10)
fig.tight_layout(); fig.savefig(FIG_DIR / "task5_prototypes.png"); plt.show()

D_fer = F.prototype_distances(landmarks("fer_gray", "train"), MEAN_SHAPE, PROTOTYPES)
proto_acc = metrics(y_tr_full, D_fer.argmin(1))
print("FER train classified by the nearest KDEF prototype alone:", {k: round(v, 3) for k, v in proto_acc.items()})
try_group("C1", {"no prototypes": {}, "+ prototype distances": {"blocks": [b for b in CFG["blocks"] if b != "proto"] + ["proto"]}})

## C2: data curation and label verification
* **Duplicates / blanks:** identical images (dHash) are kept once; when copies disagree on the
  label, all copies are dropped. Near-blank images (pixel std < 3) are dropped.
* **Label verification:** a 3-fold cross-validated logistic regression gives an out-of-fold
  prediction for every training image. An image is flagged when the model predicts another
  emotion with probability > 0.6 **and** the nearest KDEF prototype agrees with the model rather
  than with the label (two independent witnesses).
Only the training set is curated; validation and test are untouched.

In [ ]:
X_tr_img = images("fer_gray", "train")[0]
h = F.dhash(X_tr_img)
dfh = pd.DataFrame({"h": h, "y": y_tr_full, "i": np.arange(len(h))})
conflict = dfh.groupby("h").y.transform("nunique") > 1
dup_later = dfh.duplicated("h", keep="first")
blank = X_tr_img.reshape(len(X_tr_img), -1).std(1) < 3
DROP_DEDUP = np.where(conflict | dup_later | blank)[0]
print(f"curation: {int(dup_later.sum())} duplicate copies, {int(conflict.sum())} images in label-conflicting groups, "
      f"{int(blank.sum())} blank → {len(DROP_DEDUP)} dropped")

X_lr = features("fer_gray", "train", "HOG+LM")
oof = cross_val_predict(Pipeline([("s", StandardScaler()), ("lr", LogisticRegression(max_iter=2000, C=0.1))]),
                        X_lr, y_tr_full, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                        method="predict_proba", n_jobs=3)
pred_oof, conf_oof = oof.argmax(1), oof.max(1)
nearest_proto = D_fer.argmin(1)
FLAGGED = np.where((pred_oof != y_tr_full) & (conf_oof > 0.6) & (nearest_proto == pred_oof))[0]
flag_tab = pd.DataFrame({"flagged": np.bincount(y_tr_full[FLAGGED], minlength=7),
                         "train size": np.bincount(y_tr_full, minlength=7)}, index=EMO)
flag_tab["% flagged"] = 100 * flag_tab.flagged / flag_tab["train size"]
save_table(flag_tab, "task5_flagged_labels", float_format="%.1f"); display(flag_tab.round(1))
sel = FLAGGED[np.argsort(-conf_oof[FLAGGED])][:24]
show_images(X_tr_img[sel], [f"{EMO[y_tr_full[i]][:4]}→{EMO[pred_oof[i]][:4]}" for i in sel], ncols=12, size=1.25,
            suptitle="Flagged FER training labels (label → model & prototype)", fname="task5_flagged_examples.png")

try_group("C2", {"no curation": {"curation": "none"}, "dedup + blanks": {"curation": "dedup"},
                 "dedup + blanks + flagged labels": {"curation": "dedup+flags"}})

## D: merging FER and KDEF (paper's strategy)
The KDEF training subjects (never the 14 test subjects) are added to the FER training data,
either clean or degraded to look like FER images. With class weighting the weight of class
*c* is N/(K·n_c) — the paper's inverse-frequency weighting up to a constant. Rows also report the
KDEF test score: a merged model should work on **both** domains.

In [ ]:
try_group("D", {"FER only": {"merge": "none"}, "+ KDEF (clean)": {"merge": "kdef"},
                "+ KDEF (degraded to FER-like)": {"merge": "kdef_degraded"}})

## A4–A5: feature selection and final hyper-parameter tuning

In [ ]:
try_group("A4", {"no selection": {"select": "none"}, "SelectKBest (50 %)": {"select": "kbest"}, "PCA (95 % var.)": {"select": "pca95"}})

if MODEL == "SVM":
    opts = {f"C={C}": {"params": {**CFG["params"], "clf__C": C}} for C in sorted({CFG["params"].get("clf__C", 1) * f for f in (0.5, 1, 2, 4)})}
elif MODEL == "LR":
    opts = {f"C={C}": {"params": {**CFG["params"], "clf__C": C}} for C in sorted({CFG["params"].get("clf__C", 1) * f for f in (0.3, 1, 3)})}
else:
    opts = {f"trees=500, max_features={mf}": {"params": {**CFG["params"], "clf__n_estimators": 500, "clf__max_features": mf}}
            for mf in ("sqrt", 0.05)}
try_group("A5", opts)

## Ablation table and final model

In [ ]:
ab = pd.read_csv(RESULTS_DIR / "task5_ablation.csv")
abl = ab.pivot_table(index=["group", "option"], columns="split", values="macro_f1", aggfunc="first", sort=False)
abl = abl.reindex(columns=[c for c in ["val", "test", "kdef_test"] if c in abl.columns])
abl.columns = [f"macro-F1 {c}" for c in abl.columns]
save_table(abl, "task5_ablation_macro_f1"); display(abl.round(3))
print("Adopted steps:", HISTORY)
print("Final configuration:", {k: v for k, v in CFG.items() if k != "geo_idx"}, "geo_idx:", CFG["geo_idx"])

In [ ]:
t = time.time()
_, final_model, p_final = evaluate(CFG, "final", "final (full train)", base_idx=FULL_IDX, table="task5_final", max_n=FINAL_MAX_N)
print(f"final model trained in {time.time() - t:.0f}s")
spec = {"variant": CFG["variant"], "blocks": CFG["blocks"], "hog": CFG["hog"], "geo_idx": CFG["geo_idx"],
        "proto": {"mean_shape": MEAN_SHAPE, "prototypes": PROTOTYPES}, "feature": "creative", "crop_margin": 0.0}
save_bundle("final", final_model, spec, config={k: v for k, v in CFG.items() if k not in ("params",)},
            params=CFG["params"], trained_on="FER2013 train (curated/augmented per config)", task="task5")

In [ ]:
fin = pd.read_csv(RESULTS_DIR / "task5_final.csv")
t1grid = pd.read_csv(RESULTS_DIR / "task1_grid.csv")
t1b = t1grid[(t1grid.model == MODEL) & (t1grid.feature == FEAT) & (t1grid.split == "test")].iloc[0]
kit = pd.read_csv(RESULTS_DIR / "task1_baseline.csv").query("split == 'test'").iloc[0]
fr = fin[fin.split == "test"].iloc[0]
comparison = pd.DataFrame([
    {"model": "Kit baseline (RF + landmarks)", "acc": kit.acc, "bal_acc": kit.bal_acc, "macro_f1": kit.macro_f1},
    {"model": f"Task 1 best ({MODEL} + {FEAT})", "acc": t1b.acc, "bal_acc": t1b.bal_acc, "macro_f1": t1b.macro_f1},
    {"model": "Last year's group 42 (SVM, landmark-filtered test set)", "acc": 0.5516, "bal_acc": 0.4967, "macro_f1": 0.5344},
    {"model": "Creative final model", "acc": fr.acc, "bal_acc": fr.bal_acc, "macro_f1": fr.macro_f1},
]).set_index("model")
save_table(comparison, "task5_comparison"); display(comparison.round(3))

pcf = pd.DataFrame({"Task 1 best": pd.read_csv(RESULTS_DIR / "task1_per_class_f1.csv", index_col=0)[f"{MODEL}/{FEAT}"],
                    "Creative final": per_class_f1(y_te, p_final)})
save_table(pcf, "task5_per_class_f1"); display(pcf.round(3))
plot_cm(y_te, p_final, "FER test — creative final model"); plt.savefig(FIG_DIR / "task5_final_cm.png"); plt.show()